# Environment setup
Mount drive, install packages.

In [1]:
# Install dependencies
!pip install -q youtube-transcript-api transformers torch sentencepiece
import os
import json

# Mount drive and define project directory
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/lecture-summarizer'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 34.7 MB/s eta 0:00:00
Mounted at /content/drive


# Load then run BART baseline model

Load BART

In [2]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

print("Loading BART-large-CNN...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

bart_model_name = "facebook/bart-large-cnn"
bart_tokenizer = BartTokenizer.from_pretrained(bart_model_name)
bart_model = BartForConditionalGeneration.from_pretrained(bart_model_name).to(device)
bart_model.eval()  # inference mode

print(f"BART loaded. Model size: {sum(p.numel() for p in bart_model.parameters()) / 1e6:.1f}M parameters")

Loading BART-large-CNN...
Using device: cuda


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART loaded. Model size: 406.3M parameters


Build chunking + summarization function for BART model

In [3]:
def chunk_text(text, tokenizer, max_chunk_tokens=1000, overlap_tokens=50):
    """
    Split text into chunks that fit within model's token limit.

    Args:
        text: full transcript as string
        tokenizer: model's tokenizer (used for accurate token counting)
        max_chunk_tokens: max tokens per chunk (BART supports 1024; we use 1000 for safety margin)
        overlap_tokens: overlap between consecutive chunks (preserves context across boundaries)

    Returns:
        list of chunk strings
    """
    # Tokenize the entire text
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_chunk_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start += max_chunk_tokens - overlap_tokens

    return chunks


def summarize_chunk(text, model, tokenizer, max_summary_length=150, min_summary_length=40):
    """Summarize a single chunk of text."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary


def summarize_long_text(text, model, tokenizer, verbose=True):
    """
    Summarize a long document by chunking, summarizing each chunk, then concatenating.
    """
    chunks = chunk_text(text, tokenizer)
    if verbose:
        print(f"  Split into {len(chunks)} chunks")

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        if verbose:
            print(f"  Summarizing chunk {i+1}/{len(chunks)}...", end=" ", flush=True)
        summary = summarize_chunk(chunk, model, tokenizer)
        chunk_summaries.append(summary)
        if verbose:
            print(f"({len(summary.split())} words)")

    # Combine chunk summaries
    combined = ' '.join(chunk_summaries)
    return combined, chunk_summaries

Generate BART summaries of test data

In [4]:
import json
from datetime import datetime

# Define the list of lectures
lectures = [
    {
        'id': 'ml_neural_network',
        'title': 'But what is a neural network? (3Blue1Brown)',
        'domain': 'machine_learning'
    },
    {
        'id': 'history_neoliberalism',
        'title': 'Neoliberalism and the End of History - Part 3',
        'domain': 'history'
    },
    {
        'id': 'psych_wellbeing',
        'title': 'Why Having More Doesn\'t Make You Happier (Science of Well-Being)',
        'domain': 'psychology'
    },
]

# Run BART on all 3 lectures
bart_results = {}

for lec in lectures:
    print(f"\n=== {lec['title']} ===")

    # Load processed transcript
    with open(f"{PROJECT_DIR}/data/processed/{lec['id']}.txt", 'r') as f:
        transcript = f.read()

    # Generate summary
    full_summary, chunk_summaries = summarize_long_text(
        transcript, bart_model, bart_tokenizer
    )

    bart_results[lec['id']] = {
        "lecture_metadata": lec,
        "model": "facebook/bart-large-cnn",
        "config": {
            "max_chunk_tokens": 1000,
            "overlap_tokens": 50,
            "max_summary_length": 150,
            "min_summary_length": 40,
            "num_beams": 4,
        },
        "n_chunks": len(chunk_summaries),
        "chunk_summaries": chunk_summaries,
        "full_summary": full_summary,
        "summary_word_count": len(full_summary.split()),
        "timestamp": datetime.now().isoformat(),
    }

    print(f"\n  Final summary ({len(full_summary.split())} words):")
    print(f"  {full_summary[:300]}...")

# Save all results
output_path = f"{PROJECT_DIR}/data/processed/bart_baseline_summaries.json"
with open(output_path, 'w') as f:
    json.dump(bart_results, f, indent=2)

print(f"\n\nAll BART summaries saved to {output_path}")


=== But what is a neural network? (3Blue1Brown) ===
  Split into 5 chunks
  Summarizing chunk 1/5... (44 words)
  Summarizing chunk 2/5... (73 words)
  Summarizing chunk 3/5... (52 words)
  Summarizing chunk 4/5... (62 words)
  Summarizing chunk 5/5... (36 words)

  Final summary (267 words):
  A neural network can learn to recognize handwritten digits. It's meant to be loosely analogous to how networks in biological networks work. This video is just going to be devoted to the structure component of that. The following one is going to tackle learning. The network I'm showing here has alrea...

=== Neoliberalism and the End of History - Part 3 ===
  Split into 3 chunks
  Summarizing chunk 1/3... (48 words)
  Summarizing chunk 2/3... (50 words)
  Summarizing chunk 3/3... (60 words)

  Final summary (158 words):
  Noam Chomsky: Racism grew partly out of the Enlightenment. He says the racism that's surfacing is in considerable measure a reaction to the neoliberal policies of the last gene

# Load then run T5 baseline model

Loading T5 model

In [5]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

print("Loading T5-base...")

t5_model_name = "google-t5/t5-base"
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name).to(device)
t5_model.eval()

print(f"T5 loaded. Model size: {sum(p.numel() for p in t5_model.parameters()) / 1e6:.1f}M parameters")

Loading T5-base...


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5 loaded. Model size: 222.9M parameters


Building T5-specific summarization function

In [6]:
def summarize_chunk_t5(text, model, tokenizer, max_summary_length=150, min_summary_length=40):
    """Summarize a single chunk using T5 (with task prefix)."""
    # T5 requires task prefix
    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512  # T5-base has shorter context than BART
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary


def summarize_long_text_t5(text, model, tokenizer, verbose=True):
    """Summarize long document via T5 chunking."""
    # T5 has shorter context (512 tokens) so we need smaller chunks
    chunks = chunk_text(text, tokenizer, max_chunk_tokens=400, overlap_tokens=30)
    if verbose:
        print(f"  Split into {len(chunks)} chunks")

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        if verbose:
            print(f"  Summarizing chunk {i+1}/{len(chunks)}...", end=" ", flush=True)
        summary = summarize_chunk_t5(chunk, model, tokenizer)
        chunk_summaries.append(summary)
        if verbose:
            print(f"({len(summary.split())} words)")

    combined = ' '.join(chunk_summaries)
    return combined, chunk_summaries

Generate T5 summaries of test data

In [7]:
t5_results = {}

for lec in lectures:
    print(f"\n=== {lec['title']} ===")

    with open(f"{PROJECT_DIR}/data/processed/{lec['id']}.txt", 'r') as f:
        transcript = f.read()

    full_summary, chunk_summaries = summarize_long_text_t5(
        transcript, t5_model, t5_tokenizer
    )

    t5_results[lec['id']] = {
        "lecture_metadata": lec,
        "model": "google-t5/t5-base",
        "config": {
            "max_chunk_tokens": 400,
            "overlap_tokens": 30,
            "max_summary_length": 150,
            "min_summary_length": 40,
            "num_beams": 4,
            "task_prefix": "summarize: ",
        },
        "n_chunks": len(chunk_summaries),
        "chunk_summaries": chunk_summaries,
        "full_summary": full_summary,
        "summary_word_count": len(full_summary.split()),
        "timestamp": datetime.now().isoformat(),
    }

    print(f"\n  Final summary ({len(full_summary.split())} words):")
    print(f"  {full_summary[:300]}...")

output_path = f"{PROJECT_DIR}/data/processed/t5_baseline_summaries.json"
with open(output_path, 'w') as f:
    json.dump(t5_results, f, indent=2)

print(f"\n\nAll T5 summaries saved to {output_path}")


=== But what is a neural network? (3Blue1Brown) ===
  Split into 12 chunks
  Summarizing chunk 1/12... (49 words)
  Summarizing chunk 2/12... (40 words)
  Summarizing chunk 3/12... (34 words)
  Summarizing chunk 4/12... (68 words)
  Summarizing chunk 5/12... (53 words)
  Summarizing chunk 6/12... (63 words)
  Summarizing chunk 7/12... (69 words)
  Summarizing chunk 8/12... (40 words)
  Summarizing chunk 9/12... (66 words)
  Summarizing chunk 10/12... (65 words)
  Summarizing chunk 11/12... (46 words)
  Summarizing chunk 12/12... (48 words)

  Final summary (641 words):
  john avlon shows you what a neural network actually is, assuming no background . avlon: this video is going to be devoted to the structure component of a neural network . avlon: we're going to put together a video to show you what a neural network actually is . a neural network can learn to recogniz...

=== Neoliberalism and the End of History - Part 3 ===
  Split into 8 chunks
  Summarizing chunk 1/8... (33 words)
  

# Comparing baseline models: BART vs T5

In [8]:
import json

# Load both result sets
with open(f"{PROJECT_DIR}/data/processed/bart_baseline_summaries.json", 'r') as f:
    bart_results = json.load(f)

with open(f"{PROJECT_DIR}/data/processed/t5_baseline_summaries.json", 'r') as f:
    t5_results = json.load(f)

# Build comparison table
print(f"{'Lecture':<25} {'Input words':>12} {'BART':>20} {'T5':>20}")
print(f"{'':<25} {'':>12} {'words / chunks':>20} {'words / chunks':>20}")
print("-" * 80)

for lec in lectures:
    lec_id = lec['id']

    # Get input transcript word count
    with open(f"{PROJECT_DIR}/data/processed/{lec_id}.txt", 'r') as f:
        input_words = len(f.read().split())

    bart = bart_results[lec_id]
    t5 = t5_results[lec_id]

    bart_str = f"{bart['summary_word_count']} / {bart['n_chunks']}"
    t5_str = f"{t5['summary_word_count']} / {t5['n_chunks']}"

    # Truncate long lecture IDs for display
    display_name = lec_id[:24]

    print(f"{display_name:<25} {input_words:>12} {bart_str:>20} {t5_str:>20}")

# Compute compression ratios — useful summary stat
print("\n" + "=" * 80)
print("Compression ratios (summary words / input words):")
print("=" * 80)
print(f"{'Lecture':<25} {'BART ratio':>15} {'T5 ratio':>15} {'T5/BART length':>18}")
print("-" * 80)

for lec in lectures:
    lec_id = lec['id']
    with open(f"{PROJECT_DIR}/data/processed/{lec_id}.txt", 'r') as f:
        input_words = len(f.read().split())

    bart_words = bart_results[lec_id]['summary_word_count']
    t5_words = t5_results[lec_id]['summary_word_count']

    bart_ratio = bart_words / input_words
    t5_ratio = t5_words / input_words
    t5_vs_bart = t5_words / bart_words

    display_name = lec_id[:24]
    print(f"{display_name:<25} {bart_ratio:>14.1%} {t5_ratio:>14.1%} {t5_vs_bart:>17.2f}x")

Lecture                    Input words                 BART                   T5
                                             words / chunks       words / chunks
--------------------------------------------------------------------------------
ml_neural_network                 3357              267 / 5             641 / 12
history_neoliberalism             2057              158 / 3              428 / 8
psych_wellbeing                   5857              436 / 8            1130 / 21

Compression ratios (summary words / input words):
Lecture                        BART ratio        T5 ratio     T5/BART length
--------------------------------------------------------------------------------
ml_neural_network                   8.0%          19.1%              2.40x
history_neoliberalism               7.7%          20.8%              2.71x
psych_wellbeing                     7.4%          19.3%              2.59x


Save baseline model comparisons to drive

In [9]:
import csv

csv_path = f"{PROJECT_DIR}/data/processed/baseline_comparison.csv"
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'lecture_id', 'domain', 'input_words',
        'bart_summary_words', 'bart_chunks', 'bart_compression_ratio',
        't5_summary_words', 't5_chunks', 't5_compression_ratio',
        't5_vs_bart_length_factor',
    ])

    for lec in lectures:
        lec_id = lec['id']
        with open(f"{PROJECT_DIR}/data/processed/{lec_id}.txt", 'r') as fp:
            input_words = len(fp.read().split())

        bart = bart_results[lec_id]
        t5 = t5_results[lec_id]

        writer.writerow([
            lec_id,
            lec['domain'],
            input_words,
            bart['summary_word_count'],
            bart['n_chunks'],
            round(bart['summary_word_count'] / input_words, 4),
            t5['summary_word_count'],
            t5['n_chunks'],
            round(t5['summary_word_count'] / input_words, 4),
            round(t5['summary_word_count'] / bart['summary_word_count'], 2),
        ])

print(f"Saved comparison CSV to {csv_path}")

Saved comparison CSV to /content/drive/MyDrive/lecture-summarizer/data/processed/baseline_comparison.csv
